# IMF Financial Access Survey | supply-side preparation

This notebook inspects the original IMF FAS export, selects complementary normalized supply-side series, validates them, and exports an analytical long table. Run from the **notebooks** directory. The raw CSV is never modified. The year 2024 is the primary common comparison; all available historical observations remain in the export.

## 1. Imports and file paths

In [1]:
from pathlib import Path
import re
from math import isfinite
import pandas as pd
from IPython.display import display

def parse_observations(values):
    # Existing NA values and blank fields are missing, never zero.
    missing = values.isna() | values.astype("string").str.strip().eq("").fillna(False)
    parsed = pd.to_numeric(values.mask(missing), errors="coerce")
    unexpected = values.loc[~missing & parsed.isna()]
    if not unexpected.empty:
        raise ValueError(f"Unexpected nonnumeric IMF observations: {unexpected.value_counts().to_dict()}")
    observed = parsed.dropna()
    if not observed.map(isfinite).all():
        raise ValueError("Nonfinite IMF observations (+inf or -inf) require investigation")
    return parsed

RAW_PATH = Path("../data/raw/IMF_FAS_2025.csv")
OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert RAW_PATH.is_file(), f"Missing raw source: {RAW_PATH}"
raw = pd.read_csv(RAW_PATH, low_memory=False)
year_columns = sorted((c for c in raw.columns if re.fullmatch(r"(?:19|20)\d{2}", str(c))), key=int)
print(f"Source: {RAW_PATH}; shape: {raw.shape}; years: {year_columns[0]}–{year_columns[-1]}")

Source: ../data/raw/IMF_FAS_2025.csv; shape: (32017, 66); years: 2004–2025


## 2. Raw dataset inspection

In [2]:
display(raw.head(5))
print("Complete columns:", raw.columns.tolist())
display(raw.dtypes.rename("dtype").to_frame())
print("Exact duplicate rows:", raw.duplicated().sum())
display(pd.DataFrame({"missing": raw.isna().sum(), "missing_pct": raw.isna().mean().mul(100).round(1)}))

Complete columns: ['DATASET', 'SERIES_CODE', 'OBS_MEASURE', 'COUNTRY', 'INDICATOR', 'TYPE_OF_TRANSFORMATION', 'FREQUENCY', 'SCALE', 'DECIMALS_DISPLAYED', 'INSTR_ASSET', 'FA_INDICATORS', 'SECTOR', 'COUNTERPART_SECTOR', 'ACCOUNTING_ENTRY', 'SEX', 'TRANSFORMATION', 'UNIT', 'OVERLAP', 'METADATA_QUESTIONNAIRE_1', 'METADATA_QUESTIONNAIRE_2', 'DOI', 'FULL_DESCRIPTION', 'AUTHOR', 'PUBLISHER', 'DEPARTMENT', 'CONTACT_POINT', 'TOPIC', 'TOPIC_DATASET', 'KEYWORDS', 'KEYWORDS_DATASET', 'LANGUAGE', 'PUBLICATION_DATE', 'UPDATE_DATE', 'METHODOLOGY', 'METHODOLOGY_NOTES', 'ACCESS_SHARING_LEVEL', 'ACCESS_SHARING_NOTES', 'SECURITY_CLASSIFICATION', 'SHORT_SOURCE_CITATION', 'FULL_SOURCE_CITATION', 'LICENSE', 'SUGGESTED_CITATION', 'KEY_INDICATOR', 'SERIES_NAME', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']
Exact duplicate rows: 0


,DATASET,SERIES_CODE,OBS_MEASURE,COUNTRY,INDICATOR,TYPE_OF_TRANSFORMATION,FREQUENCY,SCALE,DECIMALS_DISPLAYED,INSTR_ASSET,FA_INDICATORS,SECTOR,COUNTERPART_SECTOR,ACCOUNTING_ENTRY,SEX,TRANSFORMATION,UNIT,OVERLAP,METADATA_QUESTIONNAIRE_1,METADATA_QUESTIONNAIRE_2,DOI,FULL_DESCRIPTION,AUTHOR,PUBLISHER,DEPARTMENT,CONTACT_POINT,TOPIC,TOPIC_DATASET,KEYWORDS,KEYWORDS_DATASET,LANGUAGE,PUBLICATION_DATE,UPDATE_DATE,METHODOLOGY,METHODOLOGY_NOTES,ACCESS_SHARING_LEVEL,ACCESS_SHARING_NOTES,SECURITY_CLASSIFICATION,SHORT_SOURCE_CITATION,FULL_SOURCE_CITATION,LICENSE,SUGGESTED_CITATION,KEY_INDICATOR,SERIES_NAME,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,IMF.STA:FAS(5.0.0),BRA.LOANACC_DTMFI.NUM.A,OBS_VALUE,Brazil,"Loan accounts, Deposit taking microfinance ins...",Number,Annual,Units,NaN,NaN,Loan accounts,NaN,NaN,NaN,NaN,NaN,Number,NaN,Other Depository Corporations: \n1. Commercial...,Other notes: \n1. Assets/loans; now also inclu...,NaN,The Financial Access Survey (FAS) dataset incl...,NaN,International Monetary Fund (IMF),Statistics Department (STA),datahelp@imf.org,Financial access,Financial and monetary sector,NaN,financial inclusion; financial access; use of ...,English,2026-09-25T03:21:31.161414200Z,2026-09-25T03:21:31.202399Z,NaN,NaN,Public (Unrestricted),NaN,Public or Unrestricted,Country Authorities; IMF Staff Calculations,NaN,© International Monetary Fund Copyright. All R...,International Monetary Fund. Financial Access ...,NaN,"Loan accounts, Deposit taking microfinance ins...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,IMF.STA:FAS(5.0.0),MMR.OUTL_ODT_A.XDC.A,OBS_VALUE,Myanmar,"Outstanding loans, Other deposit takers, Assets",Domestic currency,Annual,Millions,NaN,NaN,Outstanding loans,NaN,NaN,Assets,NaN,NaN,Domestic currency,NaN,NaN,"(1) In Myanmar, the financial year is April 1-...",NaN,The Financial Access Survey (FAS) dataset incl...,NaN,International Monetary Fund (IMF),Statistics Department (STA),datahelp@imf.org,Financial access,Financial and monetary sector,NaN,financial inclusion; financial access; use of ...,English,2026-09-25T03:21:31.161414200Z,2026-09-25T03:21:31.202399Z,NaN,NaN,Public (Unrestricted),NaN,Public or Unrestricted,Country Authorities; IMF Staff Calculations,NaN,© International Monetary Fund Copyright. All R...,International Monetary Fund. Financial Access ...,NaN,"Outstanding loans, Other deposit takers, Asset...",4906.04,10382.06,12541.62,18066.98,23769.35,22020.48,22151.6,36236.86,84219.65,193243.32,440277.83,568044.0,636054.0,1043364.0,1220960.35,1836435.0,2548570.69,NaN,NaN,NaN,NaN,NaN
2,IMF.STA:FAS(5.0.0),NOR.FA21_DTMFI_S14.NUM.A,OBS_VALUE,Norway,"Borrowers, Deposit taking microfinance institu...",Number,Annual,Units,NaN,NaN,Borrowers,NaN,Households,NaN,NaN,NaN,Number,NaN,1) The number of depositors: Many Norwegian ho...,1) In 2007 Norwegian banks and mortgage compan...,NaN,The Financial Access Survey (FAS) dataset incl...,NaN,International Monetary Fund (IMF),Statistics Department (STA),datahelp@imf.org,Financial access,Financial and monetary sector,NaN,financial inclusion; financial access; use of ...,English,2026-09-25T03:21:31.161414200Z,2026-09-25T03:21:31.202399Z,NaN,NaN,Public (Unrestricted),NaN,Public or Unrestricted,Country Authorities; IMF Staff Calculations,NaN,© International Monetary Fund Copyright. All R...,International Monetary Fund. Financial Access ...,NaN,"Borrowers, Deposit taking microfinance institu...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,IMF.STA:FAS(5.0.0),NOR.OUTD_CUCC_S14_L.XDC.A,OBS_VALUE,Norway,"Outstanding deposits, Credit unions and credit...",Domestic currency,Annual,Millions,NaN,Deposits,Outstanding deposits,NaN,Households,Liabilities,NaN,NaN,Domestic currency,NaN,1) The number of depositors: Many Norwegian ho...,1) In 2007 Norwegian banks and mortgage compan...,NaN,The Financial Access Survey (FAS) dataset in

,dtype
DATASET,object
SERIES_CODE,object
OBS_MEASURE,object
COUNTRY,object
INDICATOR,object
...,...
2021,float64
2022,float64
2023,float64
2024,float64


,missing,missing_pct
DATASET,0,0.0
SERIES_CODE,0,0.0
OBS_MEASURE,0,0.0
COUNTRY,0,0.0
INDICATOR,0,0.0
...,...,...
2021,20124,62.9
2022,20107,62.8
2023,20303,63.4
2024,20255,63.3


## 3. IMF structure and metadata

In [3]:
metadata_fields = ["COUNTRY","SERIES_CODE","SERIES_NAME","INDICATOR","UNIT","SCALE","SEX",
                   "SECTOR","COUNTERPART_SECTOR","INSTR_ASSET","FA_INDICATORS","FULL_DESCRIPTION"]
available_metadata = [c for c in metadata_fields if c in raw.columns]
display(raw[available_metadata].head(8))
print("Rows / distinct series codes:", len(raw), raw.SERIES_CODE.nunique())
print("Repeated source series codes:", raw.SERIES_CODE.duplicated().sum())
display(raw[available_metadata].nunique(dropna=True).rename("distinct_values").to_frame())
print("UNIT and SCALE are IMF metadata (often 'Number' and 'Units'); the denominator is in SERIES_NAME.")

Rows / distinct series codes: 32017 32017
Repeated source series codes: 0
UNIT and SCALE are IMF metadata (often 'Number' and 'Units'); the denominator is in SERIES_NAME.


,COUNTRY,SERIES_CODE,SERIES_NAME,INDICATOR,UNIT,SCALE,SEX,SECTOR,COUNTERPART_SECTOR,INSTR_ASSET,FA_INDICATORS,FULL_DESCRIPTION
0,Brazil,BRA.LOANACC_DTMFI.NUM.A,"Loan accounts, Deposit taking microfinance ins...","Loan accounts, Deposit taking microfinance ins...",Number,Units,NaN,NaN,NaN,NaN,Loan accounts,The Financial Access Survey (FAS) dataset incl...
1,Myanmar,MMR.OUTL_ODT_A.XDC.A,"Outstanding loans, Other deposit takers, Asset...","Outstanding loans, Other deposit takers, Assets",Domestic currency,Millions,NaN,NaN,NaN,NaN,Outstanding loans,The Financial Access Survey (FAS) dataset incl...
2,Norway,NOR.FA21_DTMFI_S14.NUM.A,"Borrowers, Deposit taking microfinance institu...","Borrowers, Deposit taking microfinance institu...",Number,Units,NaN,NaN,Households,NaN,Borrowers,The Financial Access Survey (FAS) dataset incl...
3,Norway,NOR.OUTD_CUCC_S14_L.XDC.A,"Outstanding deposits, Credit unions and credit...","Outstanding deposits, Credit unions and credit...",Domestic currency,Millions,NaN,NaN,Households,Deposits,Outstanding deposits,The Financial Access Survey (FAS) dataset incl...
4,Norway,NOR.FA30_CUCC.NUM.A,"Deposit accounts, Credit unions and credit coo...","Deposit accounts, Credit unions and credit coo...",Number,Units,NaN,NaN,Credit unions and credit cooperatives,NaN,Deposit accounts,The Financial Access Survey (FAS) dataset incl...
5,Norway,NOR.OUTD_COMBANK_S14_L.XDC.A,"Outstanding deposits, Commercial banks, Househ...","Outstanding deposits, Commercial banks, Househ...",Domestic currency,Millions,NaN,NaN,Households,Deposits,Outstanding deposits,The Financial Access Survey (FAS) dataset incl...
6,Brazil,BRA.FA30_DTMFI.NUM.A,"Deposit accounts, Deposit taking microfinance ...","Deposit accounts, Deposit taking microfinance ...",Number,Units,NaN,NaN,NaN,NaN,Deposit accounts,The Financial Access Survey (FAS) dataset incl...
7,Brunei Darussalam,BRN.FA21_COMBANK.NUM.A,"Borrowers, Commercial banks, Number","Borrowers, Commercial banks",Number,Units,NaN,NaN,"Deposit-taking corporations, except the centra...",NaN,Borrowers,The Financial Access Survey (FAS) dataset incl...


,distinct_values
COUNTRY,193
SERIES_CODE,32017
SERIES_NAME,191
INDICATOR,178
UNIT,3
SCALE,2
SEX,2
SECTOR,0
COUNTERPART_SECTOR,7
INSTR_ASSET,1


## 4. Filter project countries

In [4]:
roles = {
    "Egypt, Arab Republic of": "Main market", "Morocco": "Peer market",
    "Tunisia": "Peer market", "Algeria": "Peer market", "Jordan": "Peer market",
    "Saudi Arabia": "Advanced benchmark",
    "United Arab Emirates": "Supplementary benchmark",
}
main_countries = list(roles)[:6]
project = raw.loc[raw.COUNTRY.isin(roles)].copy()
assert set(project.COUNTRY) == set(roles), f"Missing: {set(roles) - set(project.COUNTRY)}"
display(project.groupby("COUNTRY").size().rename("series_rows").to_frame())
print("Project subset shape:", project.shape)

Project subset shape: (1193, 66)


,series_rows
COUNTRY,
Algeria,173
"Egypt, Arab Republic of",152
Jordan,181
Morocco,168
Saudi Arabia,174
Tunisia,172
United Arab Emirates,173


## 5. Year availability

In [5]:
year_counts = project.groupby("COUNTRY")[year_columns].apply(lambda group: group.notna().sum())
display(year_counts)
display(year_counts[["2017","2021","2024","2025"]])
print("2025 has", int(year_counts["2025"].sum()), "populated series versus",
      int(year_counts["2024"].sum()), "in 2024; Egypt 2025:", int(year_counts.loc["Egypt, Arab Republic of","2025"]))
comparison_year = 2024
assert (year_counts.loc[main_countries, str(comparison_year)] > 0).all()

2025 has 320 populated series versus 421 in 2024; Egypt 2025: 0


,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
COUNTRY,,,,,,,,,,,,,,,,,,,,,,
Algeria,23,25,25,25,25,25,25,25,27,27,27,27,27,35,35,35,43,43,51,45,47,47
"Egypt, Arab Republic of",18,22,30,30,30,30,30,34,34,40,40,46,73,73,73,79,83,83,83,83,83,0
Jordan,24,24,24,27,27,27,31,43,43,43,55,60,66,79,90,85,79,85,85,85,92,85
Morocco,29,29,29,29,29,22,22,24,24,24,24,30,30,30,30,32,36,43,39,39,39,33
Saudi Arabia,21,21,21,29,29,41,41,41,41,41,41,49,49,49,50,53,53,53,53,56,56,56
Tunisia,25,35,35,35,35,35,39,39,36,36,44,50,52,52,52,48,48,48,54,57,49,44
United Arab Emirates,19,19,19,19,19,23,23,23,23,23,55,55,55,55,55,55,55,55,55,55,55,55


,2017,2021,2024,2025
COUNTRY,,,,
Algeria,35,43,47,47
"Egypt, Arab Republic of",73,83,83,0
Jordan,79,85,92,85
Morocco,30,43,39,33
Saudi Arabia,49,53,56,56
Tunisia,52,48,49,44
United Arab Emirates,55,55,55,55


## 6. Programmatic supply-side indicator discovery

In [6]:
search_terms = ["commercial bank branches", "automated teller machines", "deposit accounts",
                "mobile money", "mobile and internet banking"]
names = project.SERIES_NAME.dropna().drop_duplicates()
for term in search_terms:
    matches = names[names.str.contains(term, case=False, regex=False)]
    print(f"\n{term}: {len(matches)} distinct series names")
    display(matches.reset_index(drop=True).to_frame("series_name"))


commercial bank branches: 2 distinct series names

automated teller machines: 3 distinct series names

deposit accounts: 17 distinct series names

mobile money: 17 distinct series names

mobile and internet banking: 4 distinct series names


,series_name
0,"Number of commercial bank branches, Per 100,00..."
1,"Number of commercial bank branches, Per 1,000 ..."


,series_name
0,"Number of automated teller machines (ATMs), Pe..."
1,"Number of automated teller machines (ATMs), Pe..."
2,"Automated teller machines (ATMs) country wide,..."


,series_name
0,"Deposit accounts, Commercial banks, Households..."
1,"Number of deposit accounts, Commercial banks, ..."
2,"Number of deposit accounts, Commercial banks, ..."
3,"Deposit accounts, Credit unions and credit coo..."
4,"Deposit accounts, Credit unions and credit coo..."
5,"Deposit accounts, Deposit taking microfinance ..."
6,"Deposit accounts, Commercial banks, Number"
7,"Deposit accounts, Credit unions and credit coo..."
8,"Deposit accounts, Deposit taking microfinance ..."
9,"Number of deposit accounts, Credit unions and ..."


,series_name
0,"Number of active mobile money agent outlets, P..."
1,"Active mobile money agent outlets, Number"
2,Mobile money transactions (during the referenc...
3,Number of mobile money transactions (during th...
4,Value of mobile money transactions (during the...
5,"Number of active mobile money agent outlets, P..."
6,Outstanding balances on active mobile money ac...
7,"Registered mobile money agent outlets, Number"
8,"Registered mobile money accounts, Number"
9,Number of registered mobile money agent outlet...


,series_name
0,Value of mobile and internet banking transacti...
1,Mobile and internet banking transactions (duri...
2,Number of mobile and internet banking transact...
3,Value of mobile and internet banking transacti...


## 7. Candidate coverage and definitions

In [7]:
candidate_names = [
    "Number of commercial bank branches, Per 100,000 adults",
    "Number of automated teller machines (ATMs), Per 100,000 adults",
    "Number of deposit accounts, Commercial banks, Per 1,000 adults",
    "Number of registered mobile money accounts, Per 1,000 adults",
    "Number of active mobile money accounts, Per 1,000 adults",
    "Number of active mobile money agent outlets, Per 100,000 adults",
    "Number of mobile money transactions (during the reference year), Per 1,000 adults",
    "Number of mobile and internet banking transactions (during the reference year), Per 1,000 adults",
    "Value of mobile money transactions (during the reference year), Percent of GDP",
    "Value of mobile and internet banking transactions (during the reference year), Percent of GDP",
]
candidates = project.loc[project.SERIES_NAME.isin(candidate_names)].copy()
assert set(candidate_names) == set(candidates.SERIES_NAME)
display(candidates[["SERIES_NAME","INDICATOR","UNIT","SCALE","SECTOR","COUNTERPART_SECTOR",
                    "SEX","INSTR_ASSET","FA_INDICATORS"]].drop_duplicates().sort_values("SERIES_NAME"))
coverage_years = ["2017","2021","2024","2025"]
candidate_long = candidates.melt(id_vars=["COUNTRY","SERIES_NAME","SERIES_CODE"], value_vars=coverage_years,
                                 var_name="year", value_name="raw_value")
candidate_long["value"] = parse_observations(candidate_long.raw_value)
coverage = candidate_long.assign(available=candidate_long.value.notna()).pivot_table(
    index=["SERIES_NAME","year"], columns="COUNTRY", values="available", aggfunc="max", fill_value=False
).reindex(columns=list(roles), fill_value=False).astype(bool)
display(coverage)
summary_2024 = coverage.xs("2024", level="year").copy()
summary_2024["main_coverage"] = summary_2024[main_countries].sum(axis=1)
summary_2024["coverage_group"] = summary_2024.main_coverage.map(
    lambda n: "All six" if n == 6 else "Missing one" if n == 5 else "Limited")
display(summary_2024.sort_values("main_coverage", ascending=False))

,SERIES_NAME,INDICATOR,UNIT,SCALE,SECTOR,COUNTERPART_SECTOR,SEX,INSTR_ASSET,FA_INDICATORS
2485,"Number of active mobile money accounts, Per 1,...",Number of active mobile money accounts,Number,Units,NaN,NaN,NaN,NaN,Number of active mobile money accounts
1539,"Number of active mobile money agent outlets, P...",Number of active mobile money agent outlets,Number,Units,NaN,NaN,NaN,NaN,Number of active mobile money agent outlets
352,"Number of automated teller machines (ATMs), Pe...",Number of automated teller machines (ATMs),Number,Units,NaN,NaN,NaN,NaN,Number of automated teller machines (ATMs)
1006,"Number of commercial bank branches, Per 100,00...",Number of commercial bank branches,Number,Units,NaN,NaN,NaN,NaN,Number of commercial bank branches
560,"Number of deposit accounts, Commercial banks, ...","Number of deposit accounts, Commercial banks",Number,Units,NaN,"Deposit-taking corporations, except the centra...",NaN,NaN,Number of deposit accounts
4277,Number of mobile and internet banking transact...,Number of mobile and internet banking transact...,Number,Units,NaN,NaN,NaN,NaN,Number of mobile and internet banking transact...
1385,Number of mobile money transactions (during th...,Number of mobile money transactions (during th...,Number,Units,NaN,NaN,NaN,NaN,Number of mobile money transactions (during th...
2508,"Number of registered mobile money accounts, Pe...",Number of registered mobile money accounts,Number,Units,NaN,NaN,NaN,NaN,Number of registered mobile money accounts
4290,Value of mobile and internet banking transacti...,Value of mobile and internet banking transacti...,NaN,Units,NaN,NaN,NaN,NaN,Value of mobile and internet banking transacti...
2515,Value of mobile money transactions (during the...,Value of mobile money transactions (during the...,NaN,Units,NaN,NaN,NaN,NaN,Value of mobile money transactions (during the...


COUNTRY                                                  Egypt, Arab Republic of  ...  United Arab Emirates
SERIES_NAME                                        year                           ...                      
Number of active mobile money accounts, Per 1,0... 2017                    False  ...                 False
                                                   2021                     True  ...                 False
                                                   2024                     True  ...                 False
                                                   2025                    False  ...                 False
Number of active mobile money agent outlets, Pe... 2017                    False  ...                 False
                                                   2021                    False  ...                 False
                                                   2024                    False  ...                 False
                                                   2025                    False  ...                 False
Number of automated teller machines (ATMs), Per... 2017                     True  ...                  True
                                                   2021                     True  ...                  True
                                                   2024                     True  ...                  True
                                                   2025                    False  ...                  True
Number of commercial bank branches, Per 100,000... 2017                     True  ...                  True
                                                   2021                     True  ...                  True
                                                   2024                     True  ...                  True
                                                   2025                    False  ...                  True
Number of deposit accounts, Commercial banks, P... 2017                     True  ...                  True
                                                   2021                     True  ...                  True
                                                   2024                     True  ...                  True
                                                   2025                    False  ...                  True
Number of mobile and internet banking transacti... 2017                    False  ...                 False
                                                   2021                     True  ...                 False
                                                   2024                     True  ...                 False
                                                   2025                    False  ...                 False
Number of mobile money transactions (during the... 2017                     True  ...                 False
                                                   2021                     True  ...                 False
                                                   2024                     True  ...                 False
                                                   2025                    False  ...                 False
Number of registered mobile money accounts, Per... 2017                     True  ...                 False
                                                   2021                     True  ...                 False
                                                   2024                     True  ...                 False
                                                   2025                    False  ...                 False
Value of mobile and internet banking transactio... 2017                    False  ...                 False
                                                   2021                     True  ...                 False
                                                   2024                     True  ...                 False
    

COUNTRY,"Egypt, Arab Republic of",Morocco,Tunisia,Algeria,Jordan,Saudi Arabia,United Arab Emirates,main_coverage,coverage_group
SERIES_NAME,,,,,,,,,
"Number of commercial bank branches, Per 100,000 adults",True,True,True,True,True,True,True,6,All six
"Number of automated teller machines (ATMs), Per 100,000 adults",True,True,True,True,True,True,True,6,All six
"Number of deposit accounts, Commercial banks, Per 1,000 adults",True,True,False,True,True,True,True,5,Missing one
"Number of registered mobile money accounts, Per 1,000 adults",True,True,True,False,True,False,False,4,Limited
"Number of mobile money transactions (during the reference year), Per 1,000 adults",True,True,True,False,True,False,False,4,Limited
"Number of mobile and internet banking transactions (during the reference year), Per 1,000 adults",True,False,True,False,True,True,False,4,Limited
"Value of mobile and internet banking transactions (during the reference year), Percent of GDP",True,False,True,False,True,True,False,4,Limited
"Value of mobile money transactions (during the reference year), Percent of GDP",True,True,True,False,True,False,False,4,Limited
"Number of active mobile money accounts, Per 1,000 adults",True,False,False,False,False,False,False,1,Limited


## 8. Final indicator selection

Branches and ATMs measure physical access across all main markets. Deposit accounts measure account intensity (accounts, **not unique people**) and omit Tunisia. The digital transaction series measures activity, with limited 2024 coverage and definitions that can differ across reporting systems. Registered mobile money accounts add a distinct supply-side product view, but registration does not imply use. These two digital series must be compared only among reporting countries; missing observations are not zero. The other candidate mobile money infrastructure and active account series have inadequate coverage. Absolute counts and local-currency values are excluded.

In [8]:
reasons = {
    candidate_names[0]: "Physical bank access; all six main markets report in 2024.",
    candidate_names[1]: "ATM access; all six main markets report in 2024.",
    candidate_names[2]: "Bank deposit account intensity; five main markets report in 2024; counts accounts, not people.",
    candidate_names[3]: "Mobile money registration supply; four main markets report in 2024; registration is not activity.",
    candidate_names[7]: "Digital banking transaction activity; four main markets report in 2024; national definitions need care.",
}
unit_map = {
    name: ("per 100,000 adults" if "Per 100,000 adults" in name else
           "per 1,000 adults" if "Per 1,000 adults" in name else "percent of GDP")
    for name in candidate_names
}
selected_names = list(reasons)
selection = pd.DataFrame([
    {"indicator_name": name, "why_selected": reason, "unit": unit_map[name],
     "available_countries_2024": ", ".join(
         ("Egypt" if c == "Egypt, Arab Republic of" else c)
         for c in roles if coverage.loc[(name,"2024"),c]),
     "historical_coverage": "; ".join(
         f"{y}: {int(coverage.loc[(name,y),main_countries].sum())}/6 main"
         for y in ["2017","2021","2024"])}
    for name,reason in reasons.items()
])
display(selection)

,indicator_name,why_selected,unit,available_countries_2024,historical_coverage
0,"Number of commercial bank branches, Per 100,00...",Physical bank access; all six main markets rep...,"per 100,000 adults","Egypt, Morocco, Tunisia, Algeria, Jordan, Saud...",2017: 6/6 main; 2021: 6/6 main; 2024: 6/6 main
1,"Number of automated teller machines (ATMs), Pe...",ATM access; all six main markets report in 2024.,"per 100,000 adults","Egypt, Morocco, Tunisia, Algeria, Jordan, Saud...",2017: 6/6 main; 2021: 6/6 main; 2024: 6/6 main
2,"Number of deposit accounts, Commercial banks, ...",Bank deposit account intensity; five main mark...,"per 1,000 adults","Egypt, Morocco, Algeria, Jordan, Saudi Arabia,...",2017: 5/6 main; 2021: 5/6 main; 2024: 5/6 main
3,"Number of registered mobile money accounts, Pe...",Mobile money registration supply; four main ma...,"per 1,000 adults","Egypt, Morocco, Tunisia, Jordan",2017: 2/6 main; 2021: 3/6 main; 2024: 4/6 main
4,Number of mobile and internet banking transact...,Digital banking transaction activity; four mai...,"per 1,000 adults","Egypt, Tunisia, Jordan, Saudi Arabia",2017: 2/6 main; 2021: 5/6 main; 2024: 4/6 main


## 9. Reshape selected series into observations

In [9]:
selected = project.loc[project.SERIES_NAME.isin(selected_names)].copy()
assert selected.groupby(["COUNTRY","SERIES_NAME"]).size().max() == 1, "Inspect duplicate source definitions"
print("Selected source series:", len(selected))
display(selected[["COUNTRY","SERIES_CODE","SERIES_NAME","INDICATOR","UNIT","SCALE",
                  "SEX","SECTOR","COUNTERPART_SECTOR"]].head(12))
long = selected.melt(id_vars=["COUNTRY","SERIES_CODE","SERIES_NAME"], value_vars=year_columns,
                     var_name="year", value_name="raw_value")
print("Reshaped rows including missing observations:", long.shape)

Selected source series: 33
Reshaped rows including missing observations: (726, 5)


,COUNTRY,SERIES_CODE,SERIES_NAME,INDICATOR,UNIT,SCALE,SEX,SECTOR,COUNTERPART_SECTOR
352,Jordan,JOR.FA18N.PHTADLT_NUM.A,"Number of automated teller machines (ATMs), Pe...",Number of automated teller machines (ATMs),Number,Units,NaN,NaN,NaN
560,"Egypt, Arab Republic of",EGY.FA30N_COMBANK.PTADLT_NUM.A,"Number of deposit accounts, Commercial banks, ...","Number of deposit accounts, Commercial banks",Number,Units,NaN,NaN,"Deposit-taking corporations, except the centra..."
577,"Egypt, Arab Republic of",EGY.FA18N.PHTADLT_NUM.A,"Number of automated teller machines (ATMs), Pe...",Number of automated teller machines (ATMs),Number,Units,NaN,NaN,NaN
1006,"Egypt, Arab Republic of",EGY.FA26N.PHTADLT_NUM.A,"Number of commercial bank branches, Per 100,00...",Number of commercial bank branches,Number,Units,NaN,NaN,NaN
2508,"Egypt, Arab Republic of",EGY.FA67N.PTADLT_NUM.A,"Number of registered mobile money accounts, Pe...",Number of registered mobile money accounts,Number,Units,NaN,NaN,NaN
4277,"Egypt, Arab Republic of",EGY.FA37N.PTADLT_NUM.A,Number of mobile and internet banking transact...,Number of mobile and internet banking transact...,Number,Units,NaN,NaN,NaN
4377,Morocco,MAR.FA37N.PTADLT_NUM.A,Number of mobile and internet banking transact...,Number of mobile and internet banking transact...,Number,Units,NaN,NaN,NaN
8095,Saudi Arabia,SAU.FA67N.PTADLT_NUM.A,"Number of registered mobile money accounts, Pe...",Number of registered mobile money accounts,Number,Units,NaN,NaN,NaN
12686,Saudi Arabia,SAU.FA26N.PHTADLT_NUM.A,"Number of commercial bank branches, Per 100,00...",Number of commercial bank branches,Number,Units,NaN,NaN,NaN
12785,Saudi Arabia,SAU.FA18N.PHTADLT_NUM.A,"Number of automated teller machines (ATMs), Pe...",Number of automated teller machines (ATMs),Number,Units,NaN,NaN,NaN


## 10. Clean and standardize

In [10]:
long["year"] = pd.to_numeric(long.year, errors="raise").astype("int64")
long["value"] = parse_observations(long.raw_value)
non_numeric = long.loc[long.raw_value.notna() & long.value.isna(), "raw_value"].value_counts()
print("Nonempty values that did not parse numerically:")
display(non_numeric.to_frame("count"))
long = long.loc[long.value.notna()].copy()
long["source_country_name"] = long.COUNTRY
long["country"] = long.COUNTRY.replace({"Egypt, Arab Republic of": "Egypt"})
long["country_code"] = long.SERIES_CODE.str.split(".").str[0]
long["indicator_name"] = long.SERIES_NAME
long["unit"] = long.indicator_name.map(unit_map)
long["source_series_code"] = long.SERIES_CODE
print("Observed rows after dropping missing values:", len(long))
assert long.unit.notna().all() and long.country_code.notna().all()

Nonempty values that did not parse numerically:
Observed rows after dropping missing values: 488


,count
raw_value,


## 11. Data quality and unusual values

In [11]:
key = ["country","year","indicator_name"]
duplicate_rows = long.loc[long.duplicated(key, keep=False)].sort_values(key)
print("Duplicate country-year-indicator observations:", len(duplicate_rows))
display(duplicate_rows.head(20))
assert duplicate_rows.empty, "Investigate dimensions and source codes before resolving duplicates"
display(long.groupby("indicator_name").value.agg(["count","min","max"]))
unusual = long.loc[(long.value < 0) | (~long.value.map(isfinite))]
print("Negative or nonfinite observations to inspect:", len(unusual))
display(unusual[["country","year","indicator_name","value","source_series_code"]].head(30))
assert long.value.map(isfinite).all(), "Nonfinite observations require investigation"
assert (long.value >= 0).all(), "Negative counts require investigation"

Duplicate country-year-indicator observations: 0
Negative or nonfinite observations to inspect: 0


,COUNTRY,SERIES_CODE,SERIES_NAME,year,raw_value,value,source_country_name,country,country_code,indicator_name,unit,source_series_code


,count,min,max
indicator_name,,,
"Number of automated teller machines (ATMs), Per 100,000 adults",149,1.264462,84.588948
"Number of commercial bank branches, Per 100,000 adults",152,3.811255,25.096291
"Number of deposit accounts, Commercial banks, Per 1,000 adults",119,307.109405,2805.270340
"Number of mobile and internet banking transactions (during the reference year), Per 1,000 adults",35,93.543431,140482.970717
"Number of registered mobile money accounts, Per 1,000 adults",33,0.046607,688.250806


,country,year,indicator_name,value,source_series_code


## 12. Country roles

In [12]:
long["country_role"] = long.source_country_name.map(roles)
assert long.country_role.notna().all(), "Unmapped country"
display(long[["country","country_role","country_code"]].drop_duplicates().sort_values("country"))

,country,country_role,country_code
20,Algeria,Peer market,DZA
2,Egypt,Main market,EGY
13,Jordan,Peer market,JOR
16,Morocco,Peer market,MAR
8,Saudi Arabia,Advanced benchmark,SAU
31,Tunisia,Peer market,TUN
28,United Arab Emirates,Supplementary benchmark,ARE


## 13. Final processed dataset validation

In [13]:
final_columns = ["country","year","indicator_name","value","unit","source_series_code",
                 "country_code","source_country_name","country_role"]
final = long[final_columns].sort_values(key, kind="stable").reset_index(drop=True)
print("Final shape:", final.shape)
display(final.head(12))
display(final.isna().sum().rename("missing").to_frame())
display(final.dtypes.rename("dtype").to_frame())
print("Countries:", sorted(final.country.unique()))
print("Years:", sorted(final.year.unique()))
display(final.groupby(["indicator_name","unit"]).value.agg(["count","min","max"]))
display(final.loc[final.year.eq(comparison_year)].pivot_table(
    index="indicator_name",columns="country",values="value",aggfunc="size",fill_value=0))
assert final[key].is_unique if hasattr(final[key], "is_unique") else not final.duplicated(key).any()
assert not final.isna().any().any()
assert set(final.indicator_name) == set(selected_names)
assert set(final.country) == {"Egypt","Morocco","Tunisia","Algeria","Jordan","Saudi Arabia","United Arab Emirates"}

Final shape: (488, 9)
Countries: ['Algeria', 'Egypt', 'Jordan', 'Morocco', 'Saudi Arabia', 'Tunisia', 'United Arab Emirates']
Years: [np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


,country,year,indicator_name,value,unit,source_series_code,country_code,source_country_name,country_role
0,Algeria,2004,"Number of automated teller machines (ATMs), Pe...",1.264462,"per 100,000 adults",DZA.FA18N.PHTADLT_NUM.A,DZA,Algeria,Peer market
1,Algeria,2004,"Number of commercial bank branches, Per 100,00...",4.717751,"per 100,000 adults",DZA.FA26N.PHTADLT_NUM.A,DZA,Algeria,Peer market
2,Algeria,2004,"Number of deposit accounts, Commercial banks, ...",623.377178,"per 1,000 adults",DZA.FA30N_COMBANK.PTADLT_NUM.A,DZA,Algeria,Peer market
3,Algeria,2005,"Number of automated teller machines (ATMs), Pe...",1.501467,"per 100,000 adults",DZA.FA18N.PHTADLT_NUM.A,DZA,Algeria,Peer market
4,Algeria,2005,"Number of commercial bank branches, Per 100,00...",4.717073,"per 100,000 adults",DZA.FA26N.PHTADLT_NUM.A,DZA,Algeria,Peer market
5,Algeria,2005,"Number of deposit accounts, Commercial banks, ...",634.424018,"per 1,000 adults",DZA.FA30N_COMBANK.PTADLT_NUM.A,DZA,Algeria,Peer market
6,Algeria,2006,"Number of automated teller machines (ATMs), Pe...",2.384927,"per 100,000 adults",DZA.FA18N.PHTADLT_NUM.A,DZA,Algeria,Peer market
7,Algeria,2006,"Number of commercial bank branches, Per 100,00...",4.944361,"per 100,000 adults",DZA.FA26N.PHTADLT_NUM.A,DZA,Algeria,Peer market
8,Algeria,2006,"Number of deposit accounts, Commercial banks, ...",666.786619,"per 1,000 adults",DZA.FA30N_COMBANK.PTADLT_NUM.A,DZA,Algeria,Peer market
9,Algeria,2007,"Number of automated teller machines (ATMs), Pe...",3.716164,"per 100,000 adults",DZA.FA18N.PHTADLT_NUM.A,DZA,Algeria,Peer market


,missing
country,0
year,0
indicator_name,0
value,0
unit,0
source_series_code,0
country_code,0
source_country_name,0
country_role,0


,dtype
country,object
year,int64
indicator_name,object
value,float64
unit,object
source_series_code,object
country_code,object
source_country_name,object
country_role,object


,,count,min,max
indicator_name,unit,,,
"Number of automated teller machines (ATMs), Per 100,000 adults","per 100,000 adults",149,1.264462,84.588948
"Number of commercial bank branches, Per 100,000 adults","per 100,000 adults",152,3.811255,25.096291
"Number of deposit accounts, Commercial banks, Per 1,000 adults","per 1,000 adults",119,307.109405,2805.270340
"Number of mobile and internet banking transactions (during the reference year), Per 1,000 adults","per 1,000 adults",35,93.543431,140482.970717
"Number of registered mobile money accounts, Per 1,000 adults","per 1,000 adults",33,0.046607,688.250806


country,Algeria,Egypt,Jordan,Morocco,Saudi Arabia,Tunisia,United Arab Emirates
indicator_name,,,,,,,
"Number of automated teller machines (ATMs), Per 100,000 adults",1,1,1,1,1,1,1
"Number of commercial bank branches, Per 100,000 adults",1,1,1,1,1,1,1
"Number of deposit accounts, Commercial banks, Per 1,000 adults",1,1,1,1,1,0,1
"Number of mobile and internet banking transactions (during the reference year), Per 1,000 adults",0,1,1,0,1,1,0
"Number of registered mobile money accounts, Per 1,000 adults",0,1,1,1,0,1,0


## 14. Export and read-back verification

In [14]:
processed_path = OUTPUT_DIR / "imf_fas_long_clean.csv"
coverage_path = OUTPUT_DIR / "imf_fas_indicator_coverage.csv"
final.to_csv(processed_path, index=False)
coverage_export = coverage.reset_index().rename(columns={"SERIES_NAME":"indicator_name"})
coverage_export["selected"] = coverage_export.indicator_name.isin(selected_names)
coverage_export.to_csv(coverage_path, index=False)
reloaded = pd.read_csv(processed_path, low_memory=False)
assert reloaded.shape == final.shape
assert reloaded.columns.tolist() == final.columns.tolist()
pd.testing.assert_frame_equal(reloaded, final, check_dtype=False, check_exact=False, rtol=1e-12)
print("Saved and verified:", processed_path, reloaded.shape)
print("Coverage summary:", coverage_path, coverage_export.shape)
display(reloaded.head())

Saved and verified: ../data/processed/imf_fas_long_clean.csv (488, 9)
Coverage summary: ../data/processed/imf_fas_indicator_coverage.csv (40, 10)


,country,year,indicator_name,value,unit,source_series_code,country_code,source_country_name,country_role
0,Algeria,2004,"Number of automated teller machines (ATMs), Pe...",1.264462,"per 100,000 adults",DZA.FA18N.PHTADLT_NUM.A,DZA,Algeria,Peer market
1,Algeria,2004,"Number of commercial bank branches, Per 100,00...",4.717751,"per 100,000 adults",DZA.FA26N.PHTADLT_NUM.A,DZA,Algeria,Peer market
2,Algeria,2004,"Number of deposit accounts, Commercial banks, ...",623.377178,"per 1,000 adults",DZA.FA30N_COMBANK.PTADLT_NUM.A,DZA,Algeria,Peer market
3,Algeria,2005,"Number of automated teller machines (ATMs), Pe...",1.501467,"per 100,000 adults",DZA.FA18N.PHTADLT_NUM.A,DZA,Algeria,Peer market
4,Algeria,2005,"Number of commercial bank branches, Per 100,00...",4.717073,"per 100,000 adults",DZA.FA26N.PHTADLT_NUM.A,DZA,Algeria,Peer market
